# AuraGateway preflight-v3 exact-runtime wheelhouse materialization v1

Materialize exactly the 196 frozen wheel identities without dependency resolution, package installation, GPU use, model loading, or benchmark execution.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import shutil
import sys
import urllib.error
import urllib.parse
import urllib.request
import zipfile
import zlib
from pathlib import Path

NOTEBOOK_NAME = "auragateway-preflight-v3-exact-runtime-wheelhouse-materialization-v1"
REQUESTED_KAGGLE_TITLE = "ag-preflight-v3-runtime-materializer-v1"
OUTPUT_DIRECTORY_NAME = "auragateway_preflight_v3_exact_runtime_wheelhouse_v1"
OUTPUT_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME
WHEELHOUSE = OUTPUT_ROOT / "wheels"
EVIDENCE_ZIP = Path("/kaggle/working") / "materialization_evidence.zip"

EXPECTED_PYTHON = (3, 12)
EXPECTED_LOCK_SHA256 = "1294394ac476336b103b036d8654a49e4ae78c25c912ca5729cd94f982384f3c"
EXPECTED_PACKAGE_COUNT = 196
EXPECTED_AUTHORITY_HOST_COUNT = 5
MAX_REDIRECTS_PER_ARTIFACT = 1
CHUNK_BYTES = 8 * 1024 * 1024

AUTHORITY_HOSTS = {
    "download-r2.pytorch.org",
    "download.pytorch.org",
    "files.pythonhosted.org",
    "github.com",
    "pypi.nvidia.com",
}
GITHUB_RELEASE_TRANSPORT_REDIRECT = (
    "github.com",
    "release-assets.githubusercontent.com",
)

CREDENTIAL_ENV_NAMES = (
    "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "GOOGLE_API_KEY",
    "HF_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
)

RESOLUTION_LOCK_ZLIB_B64 = (
    "..."
)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(CHUNK_BYTES)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: object) -> None:
    path.write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def decode_lock() -> bytes:
    payload = base64.b64decode(RESOLUTION_LOCK_ZLIB_B64.encode("ascii"))
    lock_bytes = zlib.decompress(payload)
    if sha256_bytes(lock_bytes) != EXPECTED_LOCK_SHA256:
        raise RuntimeError("EMBEDDED_RESOLUTION_LOCK_SHA256_MISMATCH")
    return lock_bytes


def normalized_host(url: str) -> str:
    parsed = urllib.parse.urlsplit(url)
    if parsed.scheme != "https":
        raise RuntimeError("DOWNLOAD_SCHEME_INVALID")
    if parsed.username is not None or parsed.password is not None:
        raise RuntimeError("DOWNLOAD_URL_CREDENTIALS_PRESENT")
    if parsed.fragment:
        raise RuntimeError("DOWNLOAD_URL_FRAGMENT_PRESENT")
    hostname = parsed.hostname
    if not isinstance(hostname, str) or not hostname:
        raise RuntimeError("DOWNLOAD_HOST_MISSING")
    return hostname.lower()


class PolicyRedirectHandler(urllib.request.HTTPRedirectHandler):
    def __init__(self) -> None:
        super().__init__()
        self.events: list[dict[str, object]] = []

    def redirect_request(
        self,
        req: urllib.request.Request,
        fp: object,
        code: int,
        msg: str,
        headers: object,
        newurl: str,
    ) -> urllib.request.Request | None:
        source_host = normalized_host(req.full_url)
        destination_host = normalized_host(newurl)

        same_host = source_host == destination_host
        github_transport = (
            source_host,
            destination_host,
        ) == GITHUB_RELEASE_TRANSPORT_REDIRECT

        if not same_host and not github_transport:
            raise RuntimeError(
                "REDIRECT_POLICY_VIOLATION:"
                f"{source_host}->{destination_host}"
            )

        self.events.append(
            {
                "source_host": source_host,
                "destination_host": destination_host,
                "status_code": int(code),
            }
        )

        if len(self.events) > MAX_REDIRECTS_PER_ARTIFACT:
            raise RuntimeError("REDIRECT_POLICY_VIOLATION:max_redirects")

        return super().redirect_request(
            req,
            fp,
            code,
            msg,
            headers,
            newurl,
        )


def download_locked_record(
    record: dict[str, object],
    index: int,
) -> dict[str, object]:
    name = record.get("normalized_name")
    filename = record.get("artifact_filename")
    locked_url = record.get("sanitized_url")
    locked_host = record.get("hostname")
    locked_sha256 = record.get("sha256")
    version = record.get("version")

    if not all(
        isinstance(value, str)
        for value in (
            name,
            filename,
            locked_url,
            locked_host,
            locked_sha256,
            version,
        )
    ):
        raise RuntimeError(f"LOCK_RECORD_INVALID:{index}")

    assert isinstance(name, str)
    assert isinstance(filename, str)
    assert isinstance(locked_url, str)
    assert isinstance(locked_host, str)
    assert isinstance(locked_sha256, str)
    assert isinstance(version, str)

    if normalized_host(locked_url) != locked_host:
        raise RuntimeError(f"LOCK_URL_HOST_MISMATCH:{index}")
    if locked_host not in AUTHORITY_HOSTS:
        raise RuntimeError(f"DOWNLOAD_HOST_DRIFT:{locked_host}")
    if not filename.lower().endswith(".whl"):
        raise RuntimeError(f"LOCK_NON_WHEEL_ARTIFACT:{index}")

    target = WHEELHOUSE / filename
    partial = WHEELHOUSE / (filename + ".part")
    if target.exists() or partial.exists():
        raise RuntimeError(f"DOWNLOAD_TARGET_ALREADY_EXISTS:{filename}")

    handler = PolicyRedirectHandler()
    opener = urllib.request.build_opener(handler)
    request = urllib.request.Request(
        locked_url,
        headers={
            "User-Agent": "auragateway-exact-runtime-materializer-v1",
            "Accept": "application/octet-stream",
        },
    )

    digest = hashlib.sha256()
    size_bytes = 0

    try:
        with opener.open(request, timeout=180) as response, partial.open("wb") as output:
            final_host = normalized_host(response.geturl())
            allowed_final_host = (
                final_host == locked_host
                or (
                    locked_host,
                    final_host,
                )
                == GITHUB_RELEASE_TRANSPORT_REDIRECT
            )
            if not allowed_final_host:
                raise RuntimeError(
                    f"DOWNLOAD_HOST_DRIFT:{locked_host}->{final_host}"
                )

            while True:
                chunk = response.read(CHUNK_BYTES)
                if not chunk:
                    break
                output.write(chunk)
                digest.update(chunk)
                size_bytes += len(chunk)
    except Exception:
        partial.unlink(missing_ok=True)
        raise

    observed_sha256 = digest.hexdigest()
    if observed_sha256 != locked_sha256:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f"DOWNLOAD_SHA256_MISMATCH:{name}:"
            f"expected={locked_sha256}:observed={observed_sha256}"
        )

    partial.replace(target)

    return {
        "record_index": index,
        "normalized_name": name,
        "version": version,
        "artifact_filename": filename,
        "locked_authority_host": locked_host,
        "sha256": observed_sha256,
        "size_bytes": size_bytes,
        "redirect_count": len(handler.events),
        "redirect_events": handler.events,
    }


if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        f"PYTHON_VERSION_MISMATCH:{sys.version_info.major}."
        f"{sys.version_info.minor}"
    )

if OUTPUT_ROOT.exists() or EVIDENCE_ZIP.exists():
    raise RuntimeError("OUTPUT_ALREADY_EXISTS")

input_root = Path("/kaggle/input")
if input_root.exists() and any(input_root.iterdir()):
    raise RuntimeError("KAGGLE_INPUTS_PRESENT")

present_credentials = [
    name for name in CREDENTIAL_ENV_NAMES if os.environ.get(name)
]
if present_credentials:
    raise RuntimeError(
        f"CREDENTIAL_ENV_PRESENT:count={len(present_credentials)}"
    )

lock_bytes = decode_lock()
lock = json.loads(lock_bytes.decode("utf-8"))
if not isinstance(lock, dict):
    raise RuntimeError("EMBEDDED_RESOLUTION_LOCK_INVALID")

records = lock.get("records")
if not isinstance(records, list) or len(records) != EXPECTED_PACKAGE_COUNT:
    raise RuntimeError("LOCK_PACKAGE_COUNT_DRIFT")
if lock.get("host_count") != EXPECTED_AUTHORITY_HOST_COUNT:
    raise RuntimeError("LOCK_AUTHORITY_HOST_COUNT_DRIFT")

host_policy = lock.get("exact_host_policy")
if not isinstance(host_policy, list):
    raise RuntimeError("LOCK_HOST_POLICY_MISSING")
observed_authority_hosts = {
    item.get("hostname")
    for item in host_policy
    if isinstance(item, dict)
}
if observed_authority_hosts != AUTHORITY_HOSTS:
    raise RuntimeError("LOCK_AUTHORITY_HOST_SET_DRIFT")

filenames = [
    record.get("artifact_filename")
    for record in records
    if isinstance(record, dict)
]
if (
    len(filenames) != EXPECTED_PACKAGE_COUNT
    or not all(isinstance(value, str) for value in filenames)
    or len(set(filenames)) != EXPECTED_PACKAGE_COUNT
):
    raise RuntimeError("LOCK_FILENAME_SET_DRIFT")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
WHEELHOUSE.mkdir(parents=True, exist_ok=False)

completed: list[dict[str, object]] = []
total_wheel_bytes = 0

try:
    for index, raw in enumerate(records):
        if not isinstance(raw, dict):
            raise RuntimeError(f"LOCK_RECORD_INVALID:{index}")

        result = download_locked_record(raw, index)
        completed.append(result)
        total_wheel_bytes += int(result["size_bytes"])

        if (index + 1) % 10 == 0 or index + 1 == EXPECTED_PACKAGE_COUNT:
            print(
                canonical_json(
                    {
                        "event": "materialization_progress",
                        "completed_count": index + 1,
                        "expected_count": EXPECTED_PACKAGE_COUNT,
                        "total_wheel_bytes": total_wheel_bytes,
                    }
                )
            )

    wheel_files = tuple(sorted(WHEELHOUSE.glob("*.whl")))
    if len(wheel_files) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("WHEEL_SET_DRIFT")

    observed_names = {path.name for path in wheel_files}
    if observed_names != set(filenames):
        raise RuntimeError("WHEEL_SET_DRIFT")

    (OUTPUT_ROOT / "resolution_lock.json").write_bytes(lock_bytes)

    requirement_lines = []
    materialization_lines = []
    for record in sorted(
        records,
        key=lambda item: str(item["normalized_name"]),
    ):
        requirement_lines.append(
            f'{record["normalized_name"]}=={record["version"]} '
            f'--hash=sha256:{record["sha256"]}'
        )
        materialization_lines.append(
            f'{record["sha256"]}  wheels/{record["artifact_filename"]}'
        )

    (OUTPUT_ROOT / "requirements.lock.txt").write_text(
        "\n".join(requirement_lines) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    (OUTPUT_ROOT / "materialization.lock.txt").write_text(
        "\n".join(materialization_lines) + "\n",
        encoding="utf-8",
        newline="\n",
    )

    redirect_events = [
        event
        for result in completed
        for event in result["redirect_events"]
    ]

    runtime_manifest = {
        "schema_version": "1.0.0",
        "manifest_id": (
            "auragateway-preflight-v3-exact-runtime-wheelhouse-runtime-manifest-v1"
        ),
        "exact_resolution_lock_sha256": EXPECTED_LOCK_SHA256,
        "locked_package_count": EXPECTED_PACKAGE_COUNT,
        "downloaded_package_count": len(completed),
        "authority_host_count": EXPECTED_AUTHORITY_HOST_COUNT,
        "authority_hosts": sorted(AUTHORITY_HOSTS),
        "transport_redirect_policy": {
            "max_redirects_per_artifact": MAX_REDIRECTS_PER_ARTIFACT,
            "github_release_source_host": GITHUB_RELEASE_TRANSPORT_REDIRECT[0],
            "github_release_destination_host": GITHUB_RELEASE_TRANSPORT_REDIRECT[1],
            "authority_host_count_unchanged": True,
        },
        "observed_redirect_event_count": len(redirect_events),
        "total_wheel_bytes": total_wheel_bytes,
        "dependency_resolution_performed": False,
        "package_installation_performed": False,
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }
    write_json(OUTPUT_ROOT / "runtime_manifest.json", runtime_manifest)

    governed_paths = [
        Path("resolution_lock.json"),
        Path("requirements.lock.txt"),
        Path("materialization.lock.txt"),
        Path("runtime_manifest.json"),
    ]
    governed_paths.extend(
        Path("wheels") / path.name
        for path in wheel_files
    )

    sha_entries = []
    for relative in governed_paths:
        path = OUTPUT_ROOT / relative
        sha_entries.append(
            {
                "path": relative.as_posix(),
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
        )

    sha_manifest = {
        "schema_version": "1.0.0",
        "manifest_id": (
            "auragateway-preflight-v3-exact-runtime-wheelhouse-sha256-manifest-v1"
        ),
        "entry_count": len(sha_entries),
        "wheel_entry_count": EXPECTED_PACKAGE_COUNT,
        "control_entry_count": 4,
        "entries": sha_entries,
    }
    write_json(OUTPUT_ROOT / "sha256_manifest.json", sha_manifest)

    receipt = {
        "schema_version": "1.0.0",
        "receipt_id": (
            "auragateway-preflight-v3-exact-runtime-wheelhouse-materialization-receipt-v1"
        ),
        "materialization_status": "PASSED_PENDING_REPOSITORY_ACCEPTANCE",
        "exact_resolution_lock_sha256": EXPECTED_LOCK_SHA256,
        "locked_package_count": EXPECTED_PACKAGE_COUNT,
        "downloaded_package_count": len(completed),
        "wheel_file_count": len(wheel_files),
        "authority_host_count": EXPECTED_AUTHORITY_HOST_COUNT,
        "observed_transport_redirect_event_count": len(redirect_events),
        "total_wheel_bytes": total_wheel_bytes,
        "sha256_manifest_sha256": sha256_file(
            OUTPUT_ROOT / "sha256_manifest.json"
        ),
        "dependency_resolution_performed": False,
        "package_installation_performed": False,
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
        "wheelhouse_materialized": True,
        "exact_runtime_materialized": False,
        "exact_runtime_offline_verified": False,
        "qualification_claimed": False,
    }
    write_json(
        OUTPUT_ROOT / "materialization_receipt.json",
        receipt,
    )

    evidence_members = (
        "resolution_lock.json",
        "requirements.lock.txt",
        "materialization.lock.txt",
        "runtime_manifest.json",
        "sha256_manifest.json",
        "materialization_receipt.json",
    )
    with zipfile.ZipFile(
        EVIDENCE_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for member in evidence_members:
            archive.write(OUTPUT_ROOT / member, arcname=member)

    print(
        canonical_json(
            {
                "materialization_status": "PASSED_PENDING_REPOSITORY_ACCEPTANCE",
                "locked_package_count": EXPECTED_PACKAGE_COUNT,
                "downloaded_package_count": len(completed),
                "wheel_file_count": len(wheel_files),
                "authority_host_count": EXPECTED_AUTHORITY_HOST_COUNT,
                "observed_transport_redirect_event_count": len(redirect_events),
                "total_wheel_bytes": total_wheel_bytes,
                "dependency_resolution_performed": False,
                "package_installation_performed": False,
                "model_loads_performed": 0,
                "model_requests_performed": 0,
                "benchmark_trajectories_performed": 0,
                "credentials_used": False,
                "customer_data_used": False,
                "external_spend": 0,
                "wheelhouse_materialized": True,
                "exact_runtime_materialized": False,
                "exact_runtime_offline_verified": False,
                "qualification_claimed": False,
                "upload_only_this_file": EVIDENCE_ZIP.name,
                "save_this_notebook_output": True,
            }
        )
    )
except Exception as error:
    failure = {
        "schema_version": "1.0.0",
        "status": "FAILED_CLOSED",
        "failure_code": type(error).__name__,
        "detail": str(error)[:500],
        "completed_download_count": len(completed),
        "expected_download_count": EXPECTED_PACKAGE_COUNT,
        "total_wheel_bytes_before_failure": total_wheel_bytes,
        "dependency_resolution_performed": False,
        "package_installation_performed": False,
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
        "qualification_claimed": False,
    }
    write_json(OUTPUT_ROOT / "materialization_failure.json", failure)
    raise
